In [ ]:
# Load necessary libraries
import sys
import os
import anndata as ad
import pandas as pd
import scanpy as sc
import seaborn as sns; sns.set(color_codes=True)
import collections
import re
from matplotlib import pyplot as plt
import random
import numpy as np
from collections import OrderedDict
import copy

random.seed(10)

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white', dpi_save=200)

In [ ]:
imageIDMapQP = {'10202021_BaharehNP_2997-ADCortex_EDTA.qptiff - resolution #1':'2997',
       '11162021_Bahareh-3155_EDTA.qptiff - resolution #1':'3155',
       '10122021_BaharehNP_ADCortex_EDTA.qptiff - resolution #1':'3196',
       '12032021-Bahareh_3280_Alz.qptiff - resolution #1':'3280',
       '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6.qptiff - resolution #1':'1796',
       '12252021_Bahareh-3628-Cntr.qptiff - resolution #1':'3628',
       '11092021_BaharehNP_3026CtlCorte.qptiff - resolution #1':'3026',
       '10222021_BaharehNP_1873_CtrlCortex.qptiff - resolution #1':'1873'}
imageTypeMapQP = {'10202021_BaharehNP_2997-ADCortex_EDTA.qptiff - resolution #1':'AD',
       '11162021_Bahareh-3155_EDTA.qptiff - resolution #1':'AD',
       '10122021_BaharehNP_ADCortex_EDTA.qptiff - resolution #1':'AD',
       '12032021-Bahareh_3280_Alz.qptiff - resolution #1':'AD',
       '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6.qptiff - resolution #1':'Ctl',
       '12252021_Bahareh-3628-Cntr.qptiff - resolution #1':'Ctl',
       '11092021_BaharehNP_3026CtlCorte.qptiff - resolution #1':'Ctl',
       '10222021_BaharehNP_1873_CtrlCortex.qptiff - resolution #1':'Ctl'}

## define variables

In [ ]:
proteins_to_cluster_with = ['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']



## load MG metadata

In [ ]:

adata = ad.read_h5ad('fabian_metadata.h5ad')
adata_orig = copy.deepcopy(adata)
adata = adata[:,proteins_to_cluster_with]
protDF = adata.to_df()

In [ ]:
#adata.obs.to_csv('data/fig7_mg_raw_fabian.csv')

In [ ]:
adata2 = adata.copy()

adata2.obs.columns

In [ ]:

adata2.obs = adata2.obs[['spatial_X', 'spatial_Y', 'area µm²', 'ImageID', 'ImageType','Parent']]
nameMap = {'Grey Matter':'GM','White Matter':'WM'}
adata2.obs.loc[:,'Region'] = adata2.obs.Parent.map(nameMap)
adata2.obs

In [ ]:
# keepCols = [
#  'TMEM119',
#  'CD74',
#  'CD14',
#  'CD163',
#  'CD68',
#  'CD11c',
#  'HLA-DR',]

keepCols = proteins_to_cluster_with
#keepCols = ['TMEM119','CD74','CD14','CD163','CD68','CD11c','HLA-DR']
# keepCols = ['TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']
# keepCols = ['TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR']


In [ ]:
[x in keepCols  for x in adata.var_names]

In [ ]:
# select only first 10 columns
#adata = adata2[:,keepCols]
adata.var['highly_variable'] = [True  for x in adata.var_names]

In [ ]:

#adataScaled=sc.pp.scale(adata, max_value=10, copy=True)
#sc.tl.pca(adata, svd_solver='arpack') # peform PCA
#sc.external.pp.bbknn(adata, batch_key='ImageID')

In [ ]:
#adataScaled = sc.pp.scale(adata, copy=True)

In [ ]:
sc.pp.log1p(adata)
#adataScaled=sc.pp.scale(adata, max_value=10, copy=True)
sc.pp.combat(adata, key='ImageID', inplace=True)
adataScaled=sc.pp.scale(adata, max_value=10, copy=True)



#sc.pp.regress_out(adataScaled,keys='spatial_Y')
#adataScaled = rescale(adata,imageid = 'ImageID', failed_markers=['CD68'], save_fig=True)

In [ ]:
adataScaled.obsm['X_spatial'] = adataScaled.obs[['spatial_X','spatial_Y']].values
adataScaled.obsm['X_spatial'][:,1] = -1*adataScaled.obsm['X_spatial'][:,1]
#adataScaled.write('../vitesscePy/QuPathDemo/1796allProt.h5ad')

In [ ]:
import sys
# set path
#sys.path.append('/data/Bahareh/')
import rapids_scanpy_funcs as rs


In [ ]:
adataScaled.var['highly_variable']

In [ ]:
sc.tl.pca(adataScaled, svd_solver='arpack', random_state = 10) # peform PCA
#sc.pl.pca(adataScaled, color=protDF.columns)
#sc.pl.pca(adataScaled, color=['ImageID'])


In [ ]:
sc.pl.pca(adataScaled, color=adata.var_names,use_raw=False)

In [ ]:
#sc.tl.tsne(adataScaled, perplexity=100,n_pcs=10,use_rep ='X_pca',use_fast_tsne=True )
sc.tl.tsne(adataScaled, perplexity=100,use_fast_tsne=True )
from cuml.manifold import TSNE
#adataScaled.obsm['X_tsne'] = TSNE(perplexity=30).fit_transform(adataScaled.obsm['X_pca'][:,:20])


In [ ]:
sc.pl.tsne(adataScaled, color=adata.var_names,use_raw=False)

In [ ]:
def fabian_style_clustering(input_adata, sample_column, copy = True, embedding_name = 'X_umap'):
    from scipy.stats import zscore
    import umap
    import umap.plot
    #from sklearn.cluster import HDBSCAN
    #from sklearn.cluster import DBSCAN

    #from sklearn.cluster import KMeans

    orig_df = input_adata.to_df()
    zscore_within_samples = input_adata.to_df().groupby(input_adata.obs[sample_column]).transform(func = zscore, axis = 0)
    zscore_all = zscore_within_samples.transform(func = zscore, axis = 0)
    
    # perform umap dimreduction
    embedding = umap.UMAP().fit(zscore_all)
    print(umap.plot.points(embedding))
    
    if copy:
        input_adata.obsm[embedding_name] = coordinates = embedding.embedding_
        return(input_adata)
    
#     coordinates = embedding.embedding_
#     coordsx = [i[0] for i in coordinates]
#     coordsy = [i[1] for i in coordinates]
    
#     coordinates_dframe = pd.DataFrame({'x':coordsx, 'y':coordsy})
#     #hdb = HDBSCAN(min_cluster_size=500, min_samples=3)
#     hdb = HDBSCAN(min_cluster_size=500, min_samples=3)
#     hdb.fit(coordinates_dframe)
    
#     kmeans = KMeans(n_clusters=6, random_state=0, n_init="auto").fit(coordinates_dframe)
#     db = DBSCAN().fit(coordinates_dframe)

    
    
    

#     print(umap.plot.points(embedding, labels = hdb.labels_))
#     print(umap.plot.points(embedding, labels = db.labels_))

#     print(umap.plot.points(embedding, labels = kmeans.labels_))

    
    
    #return(embedding)
    



adataScaled = fabian_style_clustering(adataScaled, 'ImageIDOLD')

In [ ]:
sc.pl.umap(adataScaled, color=adata.var_names,use_raw=False)

In [ ]:
# save adatascaled 
adataScaled.write_h5ad('final_h5ads/adataScaled_prior_to_clustering.h5')

In [ ]:
# add data to original metadata

adata_orig.obsm['X_tsne'] = adataScaled.obsm['X_tsne']
adata_orig.obsm['X_umap'] = adataScaled.obsm['X_umap']
adata_orig.obsm['X_pca'] = adataScaled.obsm['X_pca']

In [ ]:
#sc.pp.neighbors(adataScvarsd, n_neighbors=30, n_pcs=20) # Computing the neighborhood graph
sc.pp.neighbors(adataScaled, n_neighbors=50, n_pcs=5,use_rep ='X_pca') # Computing the neighborhood graph
#sc.pp.neighbors(adataScaled,n_neighbors=50, random_state=10) # Computing the neighborhood graph


In [ ]:
sc.pl.violin(adataScaled, groupby='ImageID', keys = 'CD163')

In [ ]:
clustering = rs.leiden(adataScaled, resolution=0.6)

collections.Counter(clustering)

In [ ]:
clustering = rs.leiden(adataScaled, resolution=0.6)

collections.Counter(clustering)

## explore inital clustering metrics

In [ ]:
def writePhenotypes(inAD, outpath, imID = 2997):
    outDF = pd.DataFrame(columns = ['ID','phenotype'])
    subAD = inAD.obs[inAD.obs['ImageIDOLD']== imID]
    subAD.reset_index(inplace=True, drop=True)
    outDF.ID = subAD.ImageID_CellID.str.split('_', expand=True)[1]
    outDF.phenotype = 'c'+subAD.leiden.astype('str')
    outDF.to_csv(outpath)


num_markers = len(keepCols)
neighbors = [50, 100, 150, 200]
resolutions = [0.2, 0.3, 0.4,0.5,0.6,0.7]

for n_neighbors in neighbors:

    sc.pp.neighbors(adataScaled,n_neighbors=n_neighbors, method='rapids', random_state=10)
    #sc.pp.neighbors(adataScaled, n_neighbors=n_neighbors, n_pcs=6,use_rep ='X_pca', random_state=10) # Computing the neighborhood graph



    from sklearn.metrics import silhouette_samples, silhouette_score
    from sklearn.metrics import calinski_harabasz_score
    range_n_clusters = resolutions
    average_score = []
    Calinski_Harabasz_scores = []
    import numpy as np
    import matplotlib.pyplot as plt
    from tqdm.notebook import tqdm
    for n_clusters in tqdm(range_n_clusters):
        mat = adataScaled.X
        visual_mat = adataScaled.obsm['X_pca']
        
        cluster_labels = rs.leiden(adataScaled, resolution=n_clusters)
        #cluster_labels = clusterer.fit_predict(X)
        # The silhouette_score gives the average value for all the samples.
        # This gives a perspective into the density and separation of the formed
        # clusters
        if len(set(cluster_labels)) == 1:
            silhouette_avg = 0
        else:
            silhouette_avg = silhouette_score(mat, cluster_labels)
        average_score.append(silhouette_avg)
        
        ch_index = calinski_harabasz_score(mat, cluster_labels)
        Calinski_Harabasz_scores.append(ch_index)






    fig, ax = plt.subplots(1,1,figsize=(15,5) )
    ax.plot(range_n_clusters, average_score)
    ax.set_xticks(range_n_clusters) 
    plt.title('mean silhouette score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()
    fig2, ax = plt.subplots(1,1,figsize=(15,5))  
    plt.plot(range_n_clusters, Calinski_Harabasz_scores)
    ax.set_xticks(range_n_clusters) 
    plt.title('Calinski Harabasz score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()


    
    

    for res in resolutions:
        

        adataScaled.obs['leiden'] = rs.leiden(adataScaled, resolution=res)

        sc.tl.dendrogram(adataScaled, groupby='leiden')
        #adataScaled.obs['State'] ='c'+adataScaled.obs['smLeiden'].astype('str')
        #sc.pl.matrixplot(adataScaled, var_names= rowOrder, groupby='leiden', 
        #                 dendrogram=True, use_raw=False, cmap="viridis",standard_scale='var' ,vmin=0.5)
        ax2 = sc.pl.matrixplot(adataScaled, var_names = protDF.columns,groupby='leiden', 
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.5,vmax=1., return_fig=True)
        sc.pl.matrixplot(adataScaled, var_names = protDF.columns,groupby='leiden', title= 'res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers),
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.50,vmax=1., return_fig=False,swap_axes=True)
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()

        sc.pl.tsne(adataScaled, color=['leiden'], title='res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers))
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()


        for image in set(adataScaled.obs['ImageIDOLD']):
            newpath = 'cluster_csvs/' + str(image) + '/' + 'n_neighbors_' + str(n_neighbors) + '/'
            if not os.path.exists(newpath):
                os.makedirs(newpath)
            
            writePhenotypes(inAD = adataScaled,outpath = (newpath + 'res_' + str(res) + '_n_neighbors_' + str(n_neighbors) + '_image_' + str(image) +  '_num_markers_' + str(num_markers) + '.csv'), imID = image)

    

    


## perform initial_clustering

In [ ]:
res = 0.2
n_neighbors = 150

# sc.tl.tsne(adataScaled, perplexity=100,use_fast_tsne=True )
# sc.pl.tsne(sc.pp.subsample(adataScaled,fraction=0.1, copy=True), color=adata.var_names,use_raw=False)

sc.pp.neighbors(adataScaled,n_neighbors=n_neighbors, method='rapids')
adataScaled.obs['leiden'] = rs.leiden(adataScaled, resolution=res)
adataScaled.obs['pre_subclustering_leiden'] = copy.deepcopy(adataScaled.obs['leiden'])



sc.tl.dendrogram(adataScaled, groupby='leiden')
ax2 = sc.pl.matrixplot(adataScaled, var_names = protDF.columns,groupby='leiden', 
                 dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.5,vmax=1., return_fig=True)
sc.pl.matrixplot(adataScaled, var_names = protDF.columns,groupby='leiden', title= 'res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers),
                 dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.50,vmax=1., return_fig=False,swap_axes=True)
plt.show()

sc.pl.tsne(adataScaled, color=['leiden'], title='res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers))
plt.show()


adataScaled.write_h5ad('cluster_csvs/' + 'res_' + str(res) + '_n_neighbors_' + str(n_neighbors) + '_num_markers_' + str(num_markers) + '.h5ad')


## explore clustering metrics for cd163+ cell subclustering

In [ ]:
adataScaled_163 = adataScaled[adataScaled.obs['leiden'] == 0]

sc.pl.tsne(adataScaled_163, color=['leiden'], title='163+ cells')
plt.show()





In [ ]:
def writePhenotypes(inAD, outpath, imID = 2997):
    outDF = pd.DataFrame(columns = ['ID','phenotype'])
    subAD = inAD.obs[inAD.obs['ImageIDOLD']== imID]
    subAD.reset_index(inplace=True, drop=True)
    outDF.ID = subAD.ImageID_CellID.str.split('_', expand=True)[1]
    outDF.phenotype = 'c'+subAD.leiden.astype('str')
    outDF.to_csv(outpath)


num_markers = len(keepCols)
neighbors = [50, 75, 100]
resolutions = [0.2, 0.3, 0.4,0.5,0.6,0.7,0.8,0.9,1,1.25,1.5,2]

for n_neighbors in neighbors:

    sc.pp.neighbors(adataScaled_163,n_neighbors=n_neighbors, method='rapids')



    from sklearn.metrics import silhouette_samples, silhouette_score
    from sklearn.metrics import calinski_harabasz_score
    range_n_clusters = resolutions
    average_score = []
    Calinski_Harabasz_scores = []
    import numpy as np
    import matplotlib.pyplot as plt
    from tqdm.notebook import tqdm
    for n_clusters in tqdm(range_n_clusters):
        mat = adataScaled_163.X
        visual_mat = adataScaled_163.obsm['X_pca']
        
    
        cluster_labels = rs.leiden(adataScaled_163, resolution=n_clusters)
        
        #cluster_labels = clusterer.fit_predict(X)
        # The silhouette_score gives the average value for all the samples.
        # This gives a perspective into the density and separation of the formed
        # clusters
        if len(set(cluster_labels)) == 1:
            silhouette_avg = 0
            ch_index = 0
        else:
            silhouette_avg = silhouette_score(mat, cluster_labels)
            ch_index = calinski_harabasz_score(mat, cluster_labels)
        Calinski_Harabasz_scores.append(ch_index)
        average_score.append(silhouette_avg)
        
        






    fig, ax = plt.subplots(1,1,figsize=(15,5) )
    ax.plot(range_n_clusters, average_score)
    ax.set_xticks(range_n_clusters) 
    plt.title('mean silhouette score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()
    fig2, ax = plt.subplots(1,1,figsize=(15,5))  
    plt.plot(range_n_clusters, Calinski_Harabasz_scores)
    ax.set_xticks(range_n_clusters) 
    plt.title('Calinski Harabasz score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()


    
    

    for res in resolutions:
        adataScaled_163.obs['leiden'] = rs.leiden(adataScaled_163, resolution=res)


        if len(set(adataScaled_163.obs['leiden'])) != 1:

            sc.tl.dendrogram(adataScaled_163, groupby='leiden')
        
        
        
        ax2 = sc.pl.matrixplot(adataScaled_163, var_names = protDF.columns,groupby='leiden', 
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.5,vmax=1., return_fig=True)
        sc.pl.matrixplot(adataScaled_163, var_names = protDF.columns,groupby='leiden', title= 'res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers),
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.50,vmax=1., return_fig=False,swap_axes=True)
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()

        sc.pl.tsne(adataScaled_163, color=['leiden'], title='res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers))
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()

       

        for image in set(adataScaled_163.obs['ImageIDOLD']):
            newpath = 'cluster_csvs_163/' + str(image) + '/' + 'n_neighbors_' + str(n_neighbors) + '/'
            if not os.path.exists(newpath):
                os.makedirs(newpath)
            
            writePhenotypes(inAD = adataScaled_163,outpath = (newpath + 'res_' + str(res) + '_n_neighbors_' + str(n_neighbors) + '_image_' + str(image) +  '_num_markers_' + str(num_markers) + '.csv'), imID = image)

    

    


## perform cd163+ cell clustering

In [ ]:
res = 0.2
n_neighbors = 50

# 
# sc.pl.tsne(sc.pp.subsample(adataScaled,fraction=0.1, copy=True), color=adata.var_names,use_raw=False)

sc.pp.neighbors(adataScaled_163,n_neighbors=n_neighbors, method='rapids')
adataScaled_163.obs['leiden'] = rs.leiden(adataScaled_163, resolution=res)

sc.tl.dendrogram(adataScaled_163, groupby='leiden')
ax2 = sc.pl.matrixplot(adataScaled_163, var_names = protDF.columns,groupby='leiden', 
                 dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.5,vmax=1., return_fig=True)
sc.pl.matrixplot(adataScaled_163, var_names = protDF.columns,groupby='leiden', title= 'res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers),
                 dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.50,vmax=1., return_fig=False,swap_axes=True)
plt.show()

sc.tl.tsne(adataScaled_163, perplexity=100,use_fast_tsne=True )

sc.pl.tsne(adataScaled_163, color=['leiden'], title='res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers))
plt.show()

adataScaled_163.write_h5ad('cluster_csvs_163/' + 'res_' + str(res) + '_n_neighbors_' + str(n_neighbors) + '_num_markers_' + str(num_markers) + '_163_positive.h5ad')





## explore clustering metrics for cd163- cell subclustering

In [ ]:
adataScaled_163_negative = adataScaled[adataScaled.obs['leiden'] == 1]

sc.pl.tsne(adataScaled_163_negative, color=['leiden'], title='163- cells')
plt.show()







In [ ]:
def writePhenotypes(inAD, outpath, imID = 2997):
    outDF = pd.DataFrame(columns = ['ID','phenotype'])
    subAD = inAD.obs[inAD.obs['ImageIDOLD']== imID]
    subAD.reset_index(inplace=True, drop=True)
    outDF.ID = subAD.ImageID_CellID.str.split('_', expand=True)[1]
    outDF.phenotype = 'c'+subAD.leiden.astype('str')
    outDF.to_csv(outpath)


num_markers = len(keepCols)
neighbors = [100,150, 200]
resolutions = [0.3, 0.4,0.5,0.6,0.7,0.8,0.9,1,1.25,1.5,2]

for n_neighbors in neighbors:

    sc.pp.neighbors(adataScaled_163_negative,n_neighbors=n_neighbors, method='rapids')



    from sklearn.metrics import silhouette_samples, silhouette_score
    from sklearn.metrics import calinski_harabasz_score
    range_n_clusters = resolutions
    average_score = []
    Calinski_Harabasz_scores = []
    import numpy as np
    import matplotlib.pyplot as plt
    from tqdm.notebook import tqdm
    for n_clusters in tqdm(range_n_clusters):
        mat = adataScaled_163_negative.X
        visual_mat = adataScaled_163_negative.obsm['X_pca']
        
    
        cluster_labels = rs.leiden(adataScaled_163_negative, resolution=n_clusters)
        #cluster_labels = clusterer.fit_predict(X)
        # The silhouette_score gives the average value for all the samples.
        # This gives a perspective into the density and separation of the formed
        # clusters
        if len(set(cluster_labels)) == 1:
            silhouette_avg = 0
        else:
            silhouette_avg = silhouette_score(mat, cluster_labels)
        average_score.append(silhouette_avg)
        
        ch_index = calinski_harabasz_score(mat, cluster_labels)
        Calinski_Harabasz_scores.append(ch_index)






    fig, ax = plt.subplots(1,1,figsize=(15,5) )
    ax.plot(range_n_clusters, average_score)
    ax.set_xticks(range_n_clusters) 
    plt.title('mean silhouette score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()
    fig2, ax = plt.subplots(1,1,figsize=(15,5))  
    plt.plot(range_n_clusters, Calinski_Harabasz_scores)
    ax.set_xticks(range_n_clusters) 
    plt.title('Calinski Harabasz score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()

    
    
    

    for res in resolutions:
        adataScaled_163_negative.obs['leiden'] = rs.leiden(adataScaled_163_negative, resolution=res)

        sc.tl.dendrogram(adataScaled_163_negative, groupby='leiden')
        
        ax2 = sc.pl.matrixplot(adataScaled_163_negative, var_names = protDF.columns,groupby='leiden', 
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.5,vmax=1., return_fig=True)
        sc.pl.matrixplot(adataScaled_163_negative, var_names = protDF.columns,groupby='leiden', title= 'res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers),
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.50,vmax=1., return_fig=False,swap_axes=True)
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()

        sc.pl.tsne(adataScaled_163_negative, color=['leiden'], title='res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers))
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()




        for image in set(adataScaled_163_negative.obs['ImageIDOLD']):
            newpath = 'cluster_csvs_163_negative/' + str(image) + '/' + 'n_neighbors_' + str(n_neighbors) + '/'
            if not os.path.exists(newpath):
                os.makedirs(newpath)
            
            writePhenotypes(inAD = adataScaled_163_negative,outpath = (newpath + 'res_' + str(res) + '_n_neighbors_' + str(n_neighbors) + '_image_' + str(image) +  '_num_markers_' + str(num_markers) + '.csv'), imID = image)

    

    


## perform cd163 - cell clustering

In [ ]:
res = 0.3
n_neighbors = 100

# 
# sc.pl.tsne(sc.pp.subsample(adataScaled,fraction=0.1, copy=True), color=adata.var_names,use_raw=False)

sc.pp.neighbors(adataScaled_163_negative,n_neighbors=n_neighbors, method='rapids')
adataScaled_163_negative.obs['leiden'] = rs.leiden(adataScaled_163_negative, resolution=res)

sc.tl.dendrogram(adataScaled_163_negative, groupby='leiden')
ax2 = sc.pl.matrixplot(adataScaled_163_negative, var_names = protDF.columns,groupby='leiden', 
                 dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.5,vmax=1., return_fig=True)
sc.pl.matrixplot(adataScaled_163_negative, var_names = protDF.columns,groupby='leiden', title= 'res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers),
                 dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.50,vmax=1., return_fig=False,swap_axes=True)
plt.show()

sc.pl.tsne(adataScaled_163_negative, color=['leiden'], title='res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers))
plt.show()




sc.tl.tsne(adataScaled_163_negative, perplexity=100,use_fast_tsne=True )

sc.pl.tsne(adataScaled_163_negative, color=['leiden'], title='res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers))
plt.show()

adataScaled_163_negative.write_h5ad('cluster_csvs_163_negative/' + 'res_' + str(res) + '_n_neighbors_' + str(n_neighbors) + '_num_markers_' + str(num_markers) + '_163_negative.h5ad')





## add subclustering metadata to full object

In [ ]:


final_cells = adataScaled_163.obs.ImageID_CellID.tolist() + adataScaled_163_negative.obs.ImageID_CellID.tolist()
adataScaled_trimmed = adataScaled[adataScaled.obs.ImageID_CellID.isin(final_cells)]

adataScaled_163_leiden = adataScaled_163.obs.leiden
adataScaled_163_leiden.index = adataScaled_163.obs['ImageID_CellID'].astype(str)
adataScaled_163_leiden_2 = 'MO_MAC_' + adataScaled_163_leiden.astype(str)

adataScaled_163_negative_leiden = adataScaled_163_negative.obs.leiden
adataScaled_163_negative_leiden.index = adataScaled_163_negative.obs['ImageID_CellID'].astype(str)
adataScaled_163_negative_leiden_2 = 'MG_' + adataScaled_163_negative_leiden.astype(str)

merged_leiden = pd.concat([adataScaled_163_leiden.astype(int), (adataScaled_163_negative_leiden.astype(int) + max(adataScaled_163_leiden.astype(int)) + 1)])
merged_leiden_2 = pd.concat([adataScaled_163_leiden_2, adataScaled_163_negative_leiden_2])

adataScaled_trimmed.obs['leiden'] = merged_leiden.loc[adataScaled_trimmed.obs.ImageID_CellID.tolist()].tolist()
adataScaled_trimmed.obs['leiden'] = adataScaled_trimmed.obs['leiden'].astype('category')

adataScaled_trimmed.obs['leiden_2'] = merged_leiden_2.loc[adataScaled_trimmed.obs.ImageID_CellID.tolist()].tolist()
adataScaled_trimmed.obs['leiden_2'] = adataScaled_trimmed.obs['leiden_2'].astype('category')

adataScaled_trimmed.write_h5ad('final_h5ads/fabian_MG_subclustering.h5ad')


## add clustering metadata to original metadata

In [ ]:

leiden_clusters = [merged_leiden_2.get(i, 'default') for i in adata_orig.obs.ImageID_CellID.tolist()]
adata_orig.obs['leiden'] = leiden_clusters

pre_subclustering_leiden_cell_dict = dict(zip(adataScaled.obs.ImageID_CellID, adataScaled.obs.pre_subclustering_leiden))
adata_orig.obs['pre_subclustering_leiden'] = [pre_subclustering_leiden_cell_dict[i] for i in adata_orig.obs.ImageID_CellID]
adata_orig.obs['pre_subclustering_leiden'] =adata_orig.obs['pre_subclustering_leiden'].astype('category')

adata_orig.write_h5ad('final_h5ads/fabian_metadata_plus_clustering.h5ad')

In [ ]:
sc.pl.tsne(adata_orig, color = 'leiden')

## make heatmap per sample

In [ ]:
# sc.pl.tsne(adataScaled_trimmed, color=['leiden_2'])

# adataScaled_final = adataScaled_trimmed

# sc.pl.matrixplot(adataScaled_final, var_names = protDF.columns,groupby='leiden_2', 
#                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=False,swap_axes=True, title='fabian_subclustering_10markers_scaled', save= 'fabian_10markers_allsamples_scaled.png')

# sc.pl.matrixplot(adataScaled_final, var_names = protDF.columns,groupby='leiden_2', 
#                     dendrogram=False, use_raw=False, cmap="inferno", standard_scale=None,vmin=-1,vmax=1, return_fig=False,swap_axes=True, title='fabian_subclustering_10markers_unscaled', save= 'fabian_10markers_allsamples_unscaled.png')


# for imID in adataScaled_final.obs.ImageIDOLD.unique():
#     subAD = adataScaled_final[adataScaled_final.obs['ImageIDOLD']== imID]
    
#     ax2 = sc.pl.matrixplot(subAD, var_names = protDF.columns,groupby='leiden_2', 
#                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True)
#     sc.pl.matrixplot(subAD, var_names = protDF.columns,groupby='leiden_2', 
#                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=False,swap_axes=True, title=str(imID), save='image_' + str(imID) + 'fabian_10markers_allsamples_scaled.png')
#     sc.pl.matrixplot(subAD, var_names = protDF.columns,groupby='leiden_2', 
#                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale=None,vmin=-1,vmax=1, return_fig=False,swap_axes=True, title=str(imID), save='image_' + str(imID) + 'fabian_10markers_allsamples_unscaled.png')




In [ ]:
# sc.pl.tsne(adataScaled_trimmed, color=['leiden_2'])

# adataScaled_final = adataScaled_trimmed

# sc.pl.matrixplot(adataScaled_final, var_names = protDF.columns,groupby='leiden_2', 
#                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=False,swap_axes=True, title='7markers', save= '7markers_0.5res_full_clustering.png')

# for imID in adataScaled_final.obs.ImageIDOLD.unique():
#     subAD = adataScaled_final[adataScaled_final.obs['ImageIDOLD']== imID]
    
#     ax2 = sc.pl.matrixplot(subAD, var_names = protDF.columns,groupby='leiden_2', 
#                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True)
#     sc.pl.matrixplot(subAD, var_names = protDF.columns,groupby='leiden_2', 
#                     dendrogram=False, use_raw=False, cmap="inferno",standard_scale='var', return_fig=False,swap_axes=True, title=str(imID), save='image_' + str(imID) + '7markers_0.5res_full_clustering.png')

# adataScaled_final.write_h5ad('final_h5ads/fabian_subclustering_ledien2.h5ad')

### perform morphological clustering

In [ ]:

adata = ad.read_h5ad('fabian_metadata.h5ad')

clustered_data_metadata = ad.read_h5ad('final_h5ads/fabian_metadata_plus_clustering.h5ad').obs

default_cluster_cells = clustered_data_metadata[clustered_data_metadata.leiden == 'default'].ImageID_CellID

adata = adata[[not i for i in adata.obs.ImageID_CellID.isin(default_cluster_cells)]]



features_to_cluster_with = ['area µm²', 'perimeter_um', 'cell_circularity',
       'cell_solidity', 'soma_area_um2', 'soma_circularity',
       'number_of_processes', 'number_of_branches', 'branching_degree',
       'average_process_length_um', 'average_process_thickness_um',
       'bi_um']


features_to_add_to_metadata = adata.obs.columns[[not i for i in adata.obs.columns.isin(features_to_cluster_with)]].to_list()

proteins_used_for_clustering = ['IBA1','TMEM119','CD74','CD14','CD163','CD68','CD11b','CD11c','HLA-DR','CD45']

adata_df = adata.to_df()

new_adata_metadata = pd.merge(adata.obs[features_to_add_to_metadata], adata_df[proteins_used_for_clustering], left_index=True, right_index=True)


new_adata_metadata['protein_leiden'] = clustered_data_metadata.loc[new_adata_metadata.ImageID_CellID].leiden
new_adata_metadata['protein_pre_subclustering_leiden'] = clustered_data_metadata.loc[new_adata_metadata.ImageID_CellID].pre_subclustering_leiden

new_adata = ad.AnnData(adata.obs[features_to_cluster_with])
new_adata.obs = new_adata_metadata

new_adata_df = new_adata.to_df()

# remove nans
new_adata = new_adata[[not i for i in new_adata_df['perimeter_um'].isna()]]

#new_adata.obs = new_adata.obs[['spatial_X', 'spatial_Y', 'area µm²', 'ImageID', 'ImageType','Parent']]
nameMap = {'Grey Matter':'GM','White Matter':'WM'}
new_adata.obs.loc[:,'Region'] = new_adata.obs.Parent.map(nameMap)









#### addy approach

In [ ]:

adata_m = copy.deepcopy(new_adata)
adata_m.layers['unscaled_morph_data'] = adata_m.to_df()

#sc.pp.log1p(adata_m)

sc.pp.combat(adata_m, key='ImageID', inplace=True)

sc.pp.scale(adata_m)

sc.tl.pca(adata_m, svd_solver='arpack') # peform PCA

sc.pl.pca(adata_m, color=adata_m.var_names,use_raw=False)


sc.tl.tsne(adata_m, perplexity=15)
# 20

sc.pl.tsne(adata_m, color=adata_m.var_names,use_raw=False)









#### explore inital clustering metrics morph

In [ ]:
def writePhenotypes(inAD, outpath, imID = 2997):
    outDF = pd.DataFrame(columns = ['ID','phenotype'])
    subAD = inAD.obs[inAD.obs['ImageIDOLD']== imID]
    subAD.reset_index(inplace=True, drop=True)
    outDF.ID = subAD.ImageID_CellID.str.split('_', expand=True)[1]
    outDF.phenotype = 'c'+subAD.leiden.astype('str')
    outDF.to_csv(outpath)

sys.path.append('/data/Bahareh/')
import rapids_scanpy_funcs as rs

num_markers = len(adata_m.var_names)
neighbors = [50, 100, 150, 200]
resolutions = [0.2, 0.3, 0.4,0.5,0.6,0.7]

for n_neighbors in neighbors:

    sc.pp.neighbors(adata_m,n_neighbors=n_neighbors, method='rapids', random_state=10)
    #sc.pp.neighbors(adata_m, n_neighbors=n_neighbors, n_pcs=6,use_rep ='X_pca', random_state=10) # Computing the neighborhood graph



    from sklearn.metrics import silhouette_samples, silhouette_score
    from sklearn.metrics import calinski_harabasz_score
    range_n_clusters = resolutions
    average_score = []
    Calinski_Harabasz_scores = []
    import numpy as np
    import matplotlib.pyplot as plt
    from tqdm.notebook import tqdm
    for n_clusters in tqdm(range_n_clusters):
        mat = adata_m.X
        visual_mat = adata_m.obsm['X_pca']
        
        cluster_labels = rs.leiden(adata_m, resolution=n_clusters)
        #cluster_labels = clusterer.fit_predict(X)
        # The silhouette_score gives the average value for all the samples.
        # This gives a perspective into the density and separation of the formed
        # clusters
        if len(set(cluster_labels)) == 1:
            silhouette_avg = 0
        else:
            silhouette_avg = silhouette_score(mat, cluster_labels)
        average_score.append(silhouette_avg)
        
        ch_index = calinski_harabasz_score(mat, cluster_labels)
        Calinski_Harabasz_scores.append(ch_index)






    fig, ax = plt.subplots(1,1,figsize=(15,5) )
    ax.plot(range_n_clusters, average_score)
    ax.set_xticks(range_n_clusters) 
    plt.title('mean silhouette score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()
    fig2, ax = plt.subplots(1,1,figsize=(15,5))  
    plt.plot(range_n_clusters, Calinski_Harabasz_scores)
    ax.set_xticks(range_n_clusters) 
    plt.title('Calinski Harabasz score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()


    
    

    for res in resolutions:
        

        adata_m.obs['leiden'] = rs.leiden(adata_m, resolution=res)

        sc.tl.dendrogram(adata_m, groupby='leiden')
        #adata_m.obs['State'] ='c'+adata_m.obs['smLeiden'].astype('str')
        #sc.pl.matrixplot(adata_m, var_names= rowOrder, groupby='leiden', 
        #                 dendrogram=True, use_raw=False, cmap="viridis",standard_scale='var' ,vmin=0.5)
        ax2 = sc.pl.matrixplot(adata_m, var_names = adata_m.var_names,groupby='leiden', 
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.5,vmax=1., return_fig=True)
        sc.pl.matrixplot(adata_m, var_names = adata_m.var_names,groupby='leiden', title= 'res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers),
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale='var', return_fig=False,swap_axes=True)
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()

        sc.pl.tsne(adata_m, color=['leiden'], title='res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers))
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()


        for image in set(adata_m.obs['ImageIDOLD']):
            newpath = 'cluster_csvs/' + str(image) + '/' + 'n_neighbors_' + str(n_neighbors) + '/'
            if not os.path.exists(newpath):
                os.makedirs(newpath)
            
            writePhenotypes(inAD = adata_m,outpath = (newpath + 'res_' + str(res) + '_n_neighbors_' + str(n_neighbors) + '_image_' + str(image) +  '_num_markers_' + str(num_markers) + '.csv'), imID = image)

    

    


#### perform initial_clustering

In [ ]:
import rapids_scanpy_funcs as rs


res = 0.2
n_neighbors = 50

# sc.tl.tsne(adata_m, perplexity=100,use_fast_tsne=True )
# sc.pl.tsne(sc.pp.subsample(adata_m,fraction=0.1, copy=True), color=adata.var_names,use_raw=False)

sc.pp.neighbors(adata_m,n_neighbors=n_neighbors, method='rapids')
adata_m.obs['leidenM1_orig'] = rs.leiden(adata_m, resolution=res)

num_markers = len(adata_m.var_names)


sc.tl.dendrogram(adata_m, groupby='leidenM1_orig')
ax2 = sc.pl.matrixplot(adata_m, var_names = adata_m.var_names,groupby='leidenM1_orig', 
                 dendrogram=True, use_raw=False, cmap="inferno",standard_scale=None,vmin=-0.5,vmax=1., return_fig=True)
sc.pl.matrixplot(adata_m, var_names = adata_m.var_names,groupby='leidenM1_orig', title= 'res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers),
                 dendrogram=True, use_raw=False, cmap="inferno",standard_scale='var', return_fig=False,swap_axes=True)
plt.show()

sc.pl.tsne(adata_m, color=['leidenM1_orig'], title='res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers))
plt.show()

adata_m.write_h5ad('cluster_csvs/' + 'res_' + str(res) + '_n_neighbors_' + str(n_neighbors) + '_num_markers_' + str(num_markers) + 'morph_.h5ad')


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import h5py


img_dict = {
    '1796': '10122021_BaharehNP_1-CtlCortex1796_EDTA_MKT6_IBA1_cell1.h5',
    '3196': '10122021_BaharehNP_ADCortex_EDTA_IBA1_cell1.h5',
    '2997': '10202021_BaharehNP_2997-ADCortex_EDTA_IBA1_cell1.h5',
    '1873': '10222021_BaharehNP_1873_CtrlCortex_IBA1_cell1.h5',
    '3026': '11092021_BaharehNP_3026CtlCorte_IBA1_cell1.h5',
    '3155': '11162021_Bahareh-3155_EDTA_IBA1_cell1.h5',
    '3280': '12032021-Bahareh_3280_Alz_IBA1_cell1.h5',
    '3628': '12252021_Bahareh-3628-Cntr_IBA1_cell1.h5'
}

def getInputs(imID):
    #print(f"Reading image file: Bahareh/GMask/{imID}_labeledMG.tif" )
    #inImg = tf.TiffFile(f'Bahareh/GMask/{imID}_labeledMG.tif').pages[0].asarray()
    
    impath = 'segmentation_files/'
    imname = img_dict[imID]
    
    with h5py.File(impath + imname, 'r') as file:
        inImg = np.array(file['matrix'])
    
    return inImg


import tifffile as tf
def getCrop(inM, keepID):
    # Find the coordinates of all non-zero pixels with the specified keepID
    non_zero_coords = np.argwhere(inM == keepID)
    # Extract the minimum and maximum x and y coordinates
    x_min, y_min = np.min(non_zero_coords, axis=0)
    x_max, y_max = np.max(non_zero_coords, axis=0)
    # Calculate width (w) and height (h) based on the min and max coordinates
    # pad by 50
    pad = 50
    #w = x_max - x_min + pad
    #h = y_max - y_min + pad
    return (inM[max(0,x_min-pad):min(x_max+pad, inM.shape[0]), max(y_min-pad,0):min(y_max+pad, inM.shape[1])]==keepID).astype(np.uint8)


def getSampleImages(adata):
    # calculate the average expression of each gene in each cluster
    cluster_means = {}
    for clusterID in np.unique(adata.obs['leidenM1']):
        cluster_cells = adata[adata.obs['leidenM1'] == clusterID].X.toarray()
        cluster_mean = np.mean(cluster_cells, axis=0)
        cluster_means[clusterID] = cluster_mean
        
    # sample four cells that are most representative of each cluster
    fig, ax = plt.subplots(4,4,figsize=(16,16))
    for ia, (clusterID, cluster_mean) in enumerate(cluster_means.items()):
        print(f"Cluster ID: {clusterID}")
        cellIDs = adata[adata.obs['leidenM1'] == clusterID].X.toarray()
        similarities = cosine_similarity(cellIDs, cluster_mean.reshape(1, -1))
        cellIDs = adata.obs[(adata.obs.leidenM1==clusterID)].iloc[np.argsort(similarities.ravel())[-4:]].ImageID_CellID.values[::-1]
        print(cellIDs)
        for ix,cID in enumerate(cellIDs):
            print(cID.split('_')[0])
            print(cID.split('_')[1])
            inIm = getInputs(cID.split('_')[0])
            inIm = getCrop(inIm, int(cID.split('_')[1]))
            ax[ia][ix].imshow(inIm)
            ax[ia][ix].set_title(f"{clusterID}_{cID}")
            
            
#getSampleImages(adata_m)

#### morph subclustering

##### 0

In [ ]:



def writePhenotypes(inAD, outpath, imID = 2997):
    outDF = pd.DataFrame(columns = ['ID','phenotype'])
    subAD = inAD.obs[inAD.obs['ImageIDOLD']== imID]
    subAD.reset_index(inplace=True, drop=True)
    outDF.ID = subAD.ImageID_CellID.str.split('_', expand=True)[1]
    outDF.phenotype = 'c'+subAD.leiden.astype('str')
    outDF.to_csv(outpath)



num_markers = len(adata_m.var_names)
neighbors = [50, 100, 150, 200]
resolutions = [0.2, 0.25, 0.3, 0.4,0.5,0.6,0.7]

cluster_to_subcluster = 0

adata_m_sub = adata_m[adata_m.obs.leidenM1_orig == cluster_to_subcluster]
print(adata_m_sub.obs.leidenM1_orig.unique())




for n_neighbors in neighbors:

    sc.pp.neighbors(adata_m_sub,n_neighbors=n_neighbors, method='rapids', random_state=10)
    #sc.pp.neighbors(adata_m_sub, n_neighbors=n_neighbors, n_pcs=6,use_rep ='X_pca', random_state=10) # Computing the neighborhood graph



    from sklearn.metrics import silhouette_samples, silhouette_score
    from sklearn.metrics import calinski_harabasz_score
    range_n_clusters = resolutions
    average_score = []
    Calinski_Harabasz_scores = []
    import numpy as np
    import matplotlib.pyplot as plt
    from tqdm.notebook import tqdm
    for n_clusters in tqdm(range_n_clusters):
        mat = adata_m_sub.X
        visual_mat = adata_m_sub.obsm['X_pca']

        cluster_labels = rs.leiden(adata_m_sub, resolution=n_clusters)
        #cluster_labels = clusterer.fit_predict(X)
        # The silhouette_score gives the average value for all the samples.
        # This gives a perspective into the density and separation of the formed
        # clusters
        if len(set(cluster_labels)) == 1:
            silhouette_avg = 0
        else:
            silhouette_avg = silhouette_score(mat, cluster_labels)
        average_score.append(silhouette_avg)

        ch_index = calinski_harabasz_score(mat, cluster_labels)
        Calinski_Harabasz_scores.append(ch_index)






    fig, ax = plt.subplots(1,1,figsize=(15,5) )
    ax.plot(range_n_clusters, average_score)
    ax.set_xticks(range_n_clusters) 
    plt.title('mean silhouette score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()
    fig2, ax = plt.subplots(1,1,figsize=(15,5))  
    plt.plot(range_n_clusters, Calinski_Harabasz_scores)
    ax.set_xticks(range_n_clusters) 
    plt.title('Calinski Harabasz score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()





    for res in resolutions:


        adata_m_sub.obs['leidenM2'] = rs.leiden(adata_m_sub, resolution=res)
        
        sc.tl.dendrogram(adata_m_sub, groupby='leidenM2')
        #adata_m_sub.obs['State'] ='c'+adata_m_sub.obs['smLeiden'].astype('str')
        #sc.pl.matrixplot(adata_m_sub, var_names= rowOrder, groupby='leiden', 
        #                 dendrogram=True, use_raw=False, cmap="viridis",standard_scale='var' ,vmin=0.5)
        ax2 = sc.pl.matrixplot(adata_m_sub, var_names = adata_m_sub.var_names,groupby='leidenM2', 
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True)
        sc.pl.matrixplot(adata_m_sub, var_names = adata_m_sub.var_names,groupby='leidenM2', title= 'res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers),
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale='var', return_fig=False,swap_axes=True)
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()

        sc.pl.tsne(adata_m_sub, color=['leidenM2'], title='res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers))
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()


        for image in set(adata_m_sub.obs['ImageIDOLD']):
            newpath = 'cluster_csvs/' + str(image) + '/' + 'n_neighbors_' + str(n_neighbors) + '/'
            if not os.path.exists(newpath):
                os.makedirs(newpath)

            writePhenotypes(inAD = adata_m_sub,outpath = (newpath + 'res_' + str(res) + '_n_neighbors_' + str(n_neighbors) + '_image_' + str(image) +  '_num_markers_' + str(num_markers) + '.csv'), imID = image)






##### 1

In [ ]:



def writePhenotypes(inAD, outpath, imID = 2997):
    outDF = pd.DataFrame(columns = ['ID','phenotype'])
    subAD = inAD.obs[inAD.obs['ImageIDOLD']== imID]
    subAD.reset_index(inplace=True, drop=True)
    outDF.ID = subAD.ImageID_CellID.str.split('_', expand=True)[1]
    outDF.phenotype = 'c'+subAD.leiden.astype('str')
    outDF.to_csv(outpath)

sys.path.append('.')
import rapids_scanpy_funcs as rs

num_markers = len(adata_m.var_names)
neighbors = [50, 100, 150, 200]
resolutions = [ 0.2, 0.3, 0.4,0.5,0.6,0.7]


cluster_to_subcluster = 1

adata_m_sub = adata_m[adata_m.obs.leidenM1_orig == cluster_to_subcluster]
print(adata_m_sub.obs.leidenM1_orig.unique())




for n_neighbors in neighbors:

    sc.pp.neighbors(adata_m_sub,n_neighbors=n_neighbors, method='rapids', random_state=10)
    #sc.pp.neighbors(adata_m_sub, n_neighbors=n_neighbors, n_pcs=6,use_rep ='X_pca', random_state=10) # Computing the neighborhood graph



    from sklearn.metrics import silhouette_samples, silhouette_score
    from sklearn.metrics import calinski_harabasz_score
    range_n_clusters = resolutions
    average_score = []
    Calinski_Harabasz_scores = []
    import numpy as np
    import matplotlib.pyplot as plt
    from tqdm.notebook import tqdm
    for n_clusters in tqdm(range_n_clusters):
        mat = adata_m_sub.X
        visual_mat = adata_m_sub.obsm['X_pca']

        cluster_labels = rs.leiden(adata_m_sub, resolution=n_clusters)
        #cluster_labels = clusterer.fit_predict(X)
        # The silhouette_score gives the average value for all the samples.
        # This gives a perspective into the density and separation of the formed
        # clusters
        if len(set(cluster_labels)) == 1:
            silhouette_avg = 0
        else:
            silhouette_avg = silhouette_score(mat, cluster_labels)
        average_score.append(silhouette_avg)

        ch_index = calinski_harabasz_score(mat, cluster_labels)
        Calinski_Harabasz_scores.append(ch_index)






    fig, ax = plt.subplots(1,1,figsize=(15,5) )
    ax.plot(range_n_clusters, average_score)
    ax.set_xticks(range_n_clusters) 
    plt.title('mean silhouette score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()
    fig2, ax = plt.subplots(1,1,figsize=(15,5))  
    plt.plot(range_n_clusters, Calinski_Harabasz_scores)
    ax.set_xticks(range_n_clusters) 
    plt.title('Calinski Harabasz score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()





    for res in resolutions:


        adata_m_sub.obs['leidenM2'] = rs.leiden(adata_m_sub, resolution=res)

        sc.tl.dendrogram(adata_m_sub, groupby='leidenM2')
        #adata_m_sub.obs['State'] ='c'+adata_m_sub.obs['smLeiden'].astype('str')
        #sc.pl.matrixplot(adata_m_sub, var_names= rowOrder, groupby='leiden', 
        #                 dendrogram=True, use_raw=False, cmap="viridis",standard_scale='var' ,vmin=0.5)
        ax2 = sc.pl.matrixplot(adata_m_sub, var_names = adata_m_sub.var_names,groupby='leidenM2', 
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True)
        sc.pl.matrixplot(adata_m_sub, var_names = adata_m_sub.var_names,groupby='leidenM2', title= 'res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers),
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale='var', return_fig=False,swap_axes=True)
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()

        sc.pl.tsne(adata_m_sub, color=['leidenM2'], title='res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers))
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()


        for image in set(adata_m_sub.obs['ImageIDOLD']):
            newpath = 'cluster_csvs/' + str(image) + '/' + 'n_neighbors_' + str(n_neighbors) + '/'
            if not os.path.exists(newpath):
                os.makedirs(newpath)

            writePhenotypes(inAD = adata_m_sub,outpath = (newpath + 'res_' + str(res) + '_n_neighbors_' + str(n_neighbors) + '_image_' + str(image) +  '_num_markers_' + str(num_markers) + '.csv'), imID = image)






##### 2

In [ ]:



def writePhenotypes(inAD, outpath, imID = 2997):
    outDF = pd.DataFrame(columns = ['ID','phenotype'])
    subAD = inAD.obs[inAD.obs['ImageIDOLD']== imID]
    subAD.reset_index(inplace=True, drop=True)
    outDF.ID = subAD.ImageID_CellID.str.split('_', expand=True)[1]
    outDF.phenotype = 'c'+subAD.leiden.astype('str')
    outDF.to_csv(outpath)

sys.path.append('/data/Bahareh/')
import rapids_scanpy_funcs as rs

num_markers = len(adata_m.var_names)
neighbors = [50, 100, 150, 200]
resolutions = [0.2, 0.3, 0.4,0.5,0.6,0.7]


cluster_to_subcluster = 2

adata_m_sub = adata_m[adata_m.obs.leidenM1_orig == cluster_to_subcluster]
print(adata_m_sub.obs.leidenM1_orig.unique())




for n_neighbors in neighbors:

    sc.pp.neighbors(adata_m_sub,n_neighbors=n_neighbors, method='rapids', random_state=10)
    #sc.pp.neighbors(adata_m_sub, n_neighbors=n_neighbors, n_pcs=6,use_rep ='X_pca', random_state=10) # Computing the neighborhood graph



    from sklearn.metrics import silhouette_samples, silhouette_score
    from sklearn.metrics import calinski_harabasz_score
    range_n_clusters = resolutions
    average_score = []
    Calinski_Harabasz_scores = []
    import numpy as np
    import matplotlib.pyplot as plt
    from tqdm.notebook import tqdm
    for n_clusters in tqdm(range_n_clusters):
        mat = adata_m_sub.X
        visual_mat = adata_m_sub.obsm['X_pca']

        cluster_labels = rs.leiden(adata_m_sub, resolution=n_clusters)
        #cluster_labels = clusterer.fit_predict(X)
        # The silhouette_score gives the average value for all the samples.
        # This gives a perspective into the density and separation of the formed
        # clusters
        if len(set(cluster_labels)) == 1:
            silhouette_avg = 0
        else:
            silhouette_avg = silhouette_score(mat, cluster_labels)
        average_score.append(silhouette_avg)

        ch_index = calinski_harabasz_score(mat, cluster_labels)
        Calinski_Harabasz_scores.append(ch_index)






    fig, ax = plt.subplots(1,1,figsize=(15,5) )
    ax.plot(range_n_clusters, average_score)
    ax.set_xticks(range_n_clusters) 
    plt.title('mean silhouette score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()
    fig2, ax = plt.subplots(1,1,figsize=(15,5))  
    plt.plot(range_n_clusters, Calinski_Harabasz_scores)
    ax.set_xticks(range_n_clusters) 
    plt.title('Calinski Harabasz score x resolution' + 'n_neighbors: ' + str(n_neighbors) + 'num markers: ' + str(num_markers))
    plt.show()





    for res in resolutions:


        adata_m_sub.obs['leidenM2'] = rs.leiden(adata_m_sub, resolution=res)

        sc.tl.dendrogram(adata_m_sub, groupby='leidenM2')
        #adata_m_sub.obs['State'] ='c'+adata_m_sub.obs['smLeiden'].astype('str')
        #sc.pl.matrixplot(adata_m_sub, var_names= rowOrder, groupby='leiden', 
        #                 dendrogram=True, use_raw=False, cmap="viridis",standard_scale='var' ,vmin=0.5)
        ax2 = sc.pl.matrixplot(adata_m_sub, var_names = adata_m_sub.var_names,groupby='leidenM2', 
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale='var', return_fig=True)
        sc.pl.matrixplot(adata_m_sub, var_names = adata_m_sub.var_names,groupby='leidenM2', title= 'res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers),
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale='var', return_fig=False,swap_axes=True)
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()

        sc.pl.tsne(adata_m_sub, color=['leidenM2'], title='res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers))
        #plt.title('res:' + str(res) + ', n_neighbors: ' + str(n_neighbors))
        plt.show()


        for image in set(adata_m_sub.obs['ImageIDOLD']):
            newpath = 'cluster_csvs/' + str(image) + '/' + 'n_neighbors_' + str(n_neighbors) + '/'
            if not os.path.exists(newpath):
                os.makedirs(newpath)

            writePhenotypes(inAD = adata_m_sub,outpath = (newpath + 'res_' + str(res) + '_n_neighbors_' + str(n_neighbors) + '_image_' + str(image) +  '_num_markers_' + str(num_markers) + '.csv'), imID = image)






#### assemble morph final clustering

In [ ]:
# subcluster ramified

cluster_to_subcluster = 0

adata_m_sub = adata_m[adata_m.obs.leidenM1_orig == cluster_to_subcluster]

res = 0.3
n_neighbors = 100

# sc.tl.tsne(adata_m, perplexity=100,use_fast_tsne=True )
# sc.pl.tsne(sc.pp.subsample(adata_m,fraction=0.1, copy=True), color=adata.var_names,use_raw=False)

sc.pp.neighbors(adata_m_sub,n_neighbors=n_neighbors, method='rapids')
adata_m_sub.obs['leidenM1_ram'] = rs.leiden(adata_m_sub, resolution=res)

sc.pl.tsne(adata_m_sub, color=['leidenM1_ram'], title='res:' + str(res) + ', n_neighbors: ' + str(n_neighbors)  + 'num markers: ' + str(num_markers))


orig_clustering_dict = dict(zip(adata_m.obs['ImageID_CellID'],adata_m.obs['leidenM1_orig']))
ramified_subclustering_dict = dict(zip(adata_m_sub.obs['ImageID_CellID'], adata_m_sub.obs['leidenM1_ram']))

for key in ramified_subclustering_dict.keys():
    orig_clustering_dict[key] = str(ramified_subclustering_dict[key]) + '_ram'


adata_m.obs['final_leiden_M1'] = [str(orig_clustering_dict[i]) for i in adata_m.obs['ImageID_CellID']]

sc.pl.matrixplot(adata_m, var_names = adata_m_sub.var_names,groupby='final_leiden_M1',
                         dendrogram=True, use_raw=False, cmap="inferno",standard_scale='var', return_fig=False,swap_axes=True)

print(collections.Counter(adata_m.obs['final_leiden_M1']))

sc.pl.tsne(adata_m, color=['final_leiden_M1'])


In [ ]:
# also add old protein tsne coords before saving

protein_clustering = ad.read_h5ad('final_h5ads/fabian_metadata_plus_clustering.h5ad')
protein_clustering.obs['X_tsne'] = [i[0] for i in protein_clustering.obsm['X_tsne']]
protein_clustering.obs['Y_tsne'] = [i[1] for i in protein_clustering.obsm['X_tsne']]

protein_clustering_sub = protein_clustering[protein_clustering.obs.ImageID_CellID.isin(adata_m.obs.ImageID_CellID)]


adata_m.obsm['X_tsne_PROTEIN'] = protein_clustering_sub.obsm['X_tsne']
sc.pl.scatter(adata_m, basis = 'tsne', color = 'final_leiden_M1')
sc.pl.scatter(adata_m, basis = 'tsne_PROTEIN', color = 'final_leiden_M1')



In [ ]:
adata_m.write_h5ad('final_h5ads/clustering_and_morph_clustering.h5')